In [1]:
# Import required libraries
import pickle  
import os      
import numpy as np  
from datetime import datetime, timedelta  
import pandas as pd  
from tqdm import tqdm  
import logging  
import time
import duckdb
from dotenv import load_dotenv

# Setup logging configuration
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Load environment variables
load_dotenv("../.env.local")
DUCKDB_PATH = os.getenv('DUCKDB_PATH')
DATA_PATH = "../DG_data/bluesky"
PROCESSED_DATA_PATH = "../processed_data/bluesky"

# Record start time for performance tracking
start_time = time.time()

# Load processed interaction data (already mapped to correct IDs)
logger.info("Loading processed interaction data...")
df = pd.read_csv(os.path.join(PROCESSED_DATA_PATH, 'ml_bluesky.csv'))

# Convert timestamps to datetime objects
df['timestamp'] = pd.to_datetime(df['ts'], unit='s')

# Load original mappings
logger.info("Loading original mappings...")
with open(os.path.join(DATA_PATH, 'post_mapping.pkl'), 'rb') as f:
    post_mapping = pickle.load(f)
with open(os.path.join(DATA_PATH, 'user_mapping.pkl'), 'rb') as f:
    user_mapping = pickle.load(f)

# Calculate the offset used for post reindexing
num_users = len(user_mapping)
logger.info(f"Number of users: {num_users}")
logger.info(f"Sample post IDs in processed data: {sorted(df['i'].unique())[:5]}")

# Determine how post IDs were reindexed in preprocessing
# In preprocessing, posts get reindexed as: original_idx + user_count + 1
post_id_offset = num_users + 1
logger.info(f"Post ID offset: {post_id_offset}")

# Connect to DuckDB
logger.info("Connecting to DuckDB...")
con = duckdb.connect(DUCKDB_PATH)

# Get post creator information from the database
logger.info("Querying post creator information...")
post_creators_df = con.execute("""
    SELECT 
        repo AS creator_did, 
        repo || '_' || rkey AS post_key,
        createdAt AS created_at
    FROM records
    WHERE createdAt >= '2023-01-01' AND collection == 'app.bsky.feed.post'
""").fetchdf()

# Convert created_at to datetime
post_creators_df['created_at'] = pd.to_datetime(post_creators_df['created_at'])

# Map post keys to their original numeric IDs
post_creators_df['original_post_id'] = post_creators_df['post_key'].map(post_mapping)
post_creators_df = post_creators_df.dropna(subset=['original_post_id'])
post_creators_df['original_post_id'] = post_creators_df['original_post_id'].astype(int)

# Map to processed post IDs by applying the offset
post_creators_df['processed_post_id'] = post_creators_df['original_post_id'] + post_id_offset

# Map creator DIDs to their numeric IDs (with +1 since IDs start from 1)
post_creators_df['creator_id'] = post_creators_df['creator_did'].map(user_mapping) + 1
post_creators_df = post_creators_df.dropna(subset=['creator_id'])
post_creators_df['creator_id'] = post_creators_df['creator_id'].astype(int)

# Create a dictionary for easier lookup (mapping post ID to tuple of creator ID and creation date)
post_creators = dict(zip(post_creators_df['processed_post_id'], 
                         zip(post_creators_df['creator_id'], post_creators_df['created_at'])))

INFO:__main__:Loading processed interaction data...
INFO:__main__:Loading original mappings...
INFO:__main__:Number of users: 106367
INFO:__main__:Sample post IDs in processed data: [106368, 106369, 106370, 106371, 106372]
INFO:__main__:Post ID offset: 106368
INFO:__main__:Connecting to DuckDB...
INFO:__main__:Querying post creator information...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [2]:
post_creators_df['processed_post_id'].min(), post_creators_df['processed_post_id'].max()

(106368, 5972593)

In [3]:
post_creators_df['creator_id'].min(), post_creators_df['creator_id'].max()

(1, 106367)

In [4]:
# Load mapped user dynamic features
logger.info("Loading user dynamic features...")
with open(os.path.join(PROCESSED_DATA_PATH, 'user_dynamic_features.pkl'), 'rb') as f:
    user_dynamic_features = pickle.load(f)

# Load producer features for initial post embeddings
logger.info("Loading producer dynamic features...")
with open(os.path.join(PROCESSED_DATA_PATH, 'producer_dynamic_features.pkl'), 'rb') as f:
    producer_dynamic_features = pickle.load(f)

# Convert date keys to datetime for easier lookup
date_to_timestamp = {datetime.fromtimestamp(ts).date(): ts for ts in user_dynamic_features.keys()}

# Add embedding date column (day of interaction) for temporal alignment
df['embedding_date'] = df['timestamp'].dt.date

# Prepare data for processing
logger.info("Preparing data...")
df = df.sort_values(['i', 'timestamp'])  # Sort by post and time
df = df[(df['timestamp'] >= '2023-03-15') & (df['timestamp'] <= '2023-03-30')]  # Filter by time range
# df = df[(df['timestamp'] >= '2023-06-01')]
grouped_posts = df.groupby('i')  # Group by post ID (already mapped)

# Initialize list to store embedding results
all_embeddings = []

# Stats tracking
missing_user_count = 0
processed_posts = 0
skipped_posts = 0
initial_embeddings_count = 0
missing_creator_count = 0

# Process each post's interactions
logger.info("Processing posts...")
embedding_dim = 64  # Adjust based on your actual embedding dimension

for post_id, post_interactions in tqdm(grouped_posts):
    try:
        # Sort interactions by timestamp to ensure chronological order
        post_interactions = post_interactions.sort_values('timestamp')
        
        # Initialize running average variables
        running_sum = None
        count = 0
        
        # Get the initial embedding for this post if available
        creator_info = post_creators.get(post_id)
        initial_embedding = None
        embedding_source = None
        
        if creator_info:
            creator_id, created_at = creator_info
            creation_date = created_at.date()
            
            if creation_date in date_to_timestamp:
                date_timestamp = date_to_timestamp[creation_date]
                
                # Try to get the producer embedding first
                if creator_id in producer_dynamic_features.get(date_timestamp, {}):
                    initial_embedding = producer_dynamic_features[date_timestamp][creator_id]
                    embedding_source = "producer"
                # Fall back to consumer embedding
                elif creator_id in user_dynamic_features.get(date_timestamp, {}):
                    initial_embedding = user_dynamic_features[date_timestamp][creator_id]
                    embedding_source = "consumer"
                else:
                    # If no embedding found, use zeros
                    initial_embedding = np.zeros(embedding_dim, dtype=np.float32)
                    embedding_source = "zero"
                
                # Initialize running sum with initial embedding
                if initial_embedding is not None:
                    running_sum = initial_embedding.copy()
                    count = 1
                    
                    # Add the initial embedding to the results
                    all_embeddings.append({
                        'post_id': int(post_id),
                        'user_id': creator_id,
                        'timestamp': created_at,
                        'embedding': initial_embedding.astype(np.float32),
                        'num_interactions': 0,  # 0 interactions
                        'embedding_source': embedding_source
                    })
                    initial_embeddings_count += 1
        else:
            missing_creator_count += 1
        
        # Initialize running sum if not already done
        if running_sum is None:
            running_sum = np.zeros(embedding_dim, dtype=np.float32)
        
        # Process each interaction in chronological order
        for i, row in post_interactions.iterrows():
            try:
                user_id = int(row['u'])  # User ID from processed data
                interaction_date = row['embedding_date']
                
                # Find the closest date in user_dynamic_features
                if interaction_date in date_to_timestamp:
                    date_timestamp = date_to_timestamp[interaction_date]
                    
                    if user_id in user_dynamic_features[date_timestamp]:
                        # Get user embedding for this interaction
                        user_embedding = user_dynamic_features[date_timestamp][user_id]
                        
                        # Update running sum and count
                        running_sum += user_embedding
                        count += 1
                        
                        # Calculate average using running sum
                        avg_embedding = running_sum / count
                        
                        all_embeddings.append({
                            'post_id': int(post_id),
                            'user_id': user_id,
                            'timestamp': row['timestamp'],
                            'embedding': avg_embedding.astype(np.float32),
                            'num_interactions': count - (1 if initial_embedding is not None else 0),  # Count interactions only
                            'embedding_source': 'avg'  # This is now an average
                        })
                    else:
                        missing_user_count += 1
            except Exception as e:
                logger.error(f"Error processing interaction for post {post_id}, user {row['u']}: {str(e)}")
                continue
        
        processed_posts += 1
                        
    except Exception as e:
        logger.error(f"Error processing post {post_id}: {str(e)}")
        continue

logger.info(f"Processed {processed_posts} posts with {missing_user_count} missing users")
logger.info(f"Created {initial_embeddings_count} initial embeddings and {len(all_embeddings) - initial_embeddings_count} interaction embeddings")
logger.info(f"Missing creators for {missing_creator_count} posts")
logger.info(f"Processing completed in {time.time() - start_time:.2f} seconds")

# Save as parquet
post_embeddings_df = pd.DataFrame(all_embeddings)
post_embeddings_df = post_embeddings_df.sort_values('timestamp')
output_file = os.path.join(os.path.expanduser("~"), 'post_dynamic_embeddings.parquet')
post_embeddings_df.to_parquet(output_file, compression='snappy')

# Verify embeddings
logger.info("Loading embeddings to verify...")
post_embeddings_path = os.path.join(os.path.expanduser("~"), 'post_dynamic_embeddings.parquet')
post_embeddings = pd.read_parquet(post_embeddings_path)
logger.info(f"Successfully loaded {len(post_embeddings)} post embeddings")

INFO:__main__:Loading user dynamic features...
INFO:__main__:Loading producer dynamic features...
INFO:__main__:Preparing data...
INFO:__main__:Processing posts...
100%|██████████| 21920/21920 [00:07<00:00, 3077.86it/s]
INFO:__main__:Processed 21920 posts with 0 missing users
INFO:__main__:Created 15635 initial embeddings and 59925 interaction embeddings
INFO:__main__:Missing creators for 5189 posts
INFO:__main__:Processing completed in 69.46 seconds
INFO:__main__:Loading embeddings to verify...
INFO:__main__:Successfully loaded 75560 post embeddings


In [5]:
# Check how many embeddings are zero vectors
zero_vectors = 0
for _, row in post_embeddings_df.iterrows():
    if np.all(row['embedding'] == 0):
        zero_vectors += 1

print(f"Total embeddings: {len(post_embeddings_df)}")
print(f"Zero vector embeddings: {zero_vectors} ({zero_vectors/len(post_embeddings_df)*100:.2f}%)")

# Display the dataframe
post_embeddings_df

Total embeddings: 75560
Zero vector embeddings: 3957 (5.24%)


,post_id,user_id,timestamp,embedding,num_interactions,embedding_source
27025,1083354,28292,2023-03-15 00:00:00.000,"[-0.19816363, 0.17425328, -0.12171154, -0.5630...",1,avg
27032,1083355,28292,2023-03-15 00:00:00.000,"[-0.19816363, 0.17425328, -0.12171154, -0.5630...",1,avg
50542,2374256,28292,2023-03-15 00:00:08.000,"[-0.19816363, 0.17425328, -0.12171154, -0.5630...",1,avg
72145,5450300,32529,2023-03-15 00:00:13.305,"[-0.17123769, 0.20906343, 0.09962787, 0.398206...",0,producer
66949,4254353,59091,2023-03-15 00:00:41.000,"[-0.22422381, 0.30928805, -0.04585657, 0.57327...",1,avg
...,...,...,...,...,...,...
51482,2434500,23920,2023-03-29 23:58:40.000,"[-0.16260566, 0.1134752, -0.08472568, -0.36419...",2,avg
39883,1488785,39585,2023-03-29 23:58:41.000,"[-0.2241105, 0.34822512, 0.023017205, 0.431257...",3,avg
11259,373731,15426,2023-03-29 23:59:10.000,"[-0.2996875, 0.04072892, -0.14770992, -0.15222...",3,avg
24720,963307,46329,2023-03-29 23:59:12.000,"[-0.30005148, 0.028809072, -0.10447627, -0.263...",16,avg


In [6]:
df

,Unnamed: 0,u,i,ts,label,idx,timestamp,embedding_date
145672,145672,68476,107458,1.679095e+09,0.0,145673,2023-03-17 23:24:41,2023-03-17
168964,168964,63108,107459,1.679588e+09,0.0,168965,2023-03-23 16:12:47,2023-03-23
172110,172110,41007,107499,1.679664e+09,0.0,172111,2023-03-24 13:13:40,2023-03-24
172124,172124,9302,107499,1.679664e+09,0.0,172125,2023-03-24 13:17:47,2023-03-24
132290,132290,17333,107660,1.678845e+09,0.0,132291,2023-03-15 01:50:33,2023-03-15
...,...,...,...,...,...,...,...,...
168938,168938,105921,5962623,1.679587e+09,0.0,168939,2023-03-23 16:03:22,2023-03-23
179480,179480,105921,5962624,1.679861e+09,0.0,179481,2023-03-26 20:04:28,2023-03-26
184656,184656,105921,5962625,1.679990e+09,0.0,184657,2023-03-28 07:58:32,2023-03-28
152277,152277,106023,5967758,1.679229e+09,0.0,152278,2023-03-19 12:33:46,2023-03-19


In [7]:
post_embeddings_df['embedding_source'].value_counts()

embedding_source
avg         59925
producer    15567
consumer       45
zero           23
Name: count, dtype: int64

In [14]:
post_creators_df

,creator_did,post_key,created_at,original_post_id,processed_post_id,creator_id
0,did:plc:3sa4qjux2g54xjn4mi35uiiu,did:plc:3sa4qjux2g54xjn4mi35uiiu_3jssezoaaks22,2023-04-07 17:56:39.081,1383574,1489942,41063
2,did:plc:3sa4qjux2g54xjn4mi35uiiu,did:plc:3sa4qjux2g54xjn4mi35uiiu_3jssgreu7s327,2023-04-07 18:27:48.346,1383575,1489943,41063
4,did:plc:3sa4qjux2g54xjn4mi35uiiu,did:plc:3sa4qjux2g54xjn4mi35uiiu_3jssums6b7m2k,2023-04-07 22:35:47.279,356490,462858,41063
6,did:plc:3sa4qjux2g54xjn4mi35uiiu,did:plc:3sa4qjux2g54xjn4mi35uiiu_3jssy7xqvdt27,2023-04-07 23:40:11.776,1383627,1489995,41063
9,did:plc:3sa4qjux2g54xjn4mi35uiiu,did:plc:3sa4qjux2g54xjn4mi35uiiu_3jsszn7pcje2k,2023-04-08 00:05:29.895,5309476,5415844,41063
...,...,...,...,...,...,...
7573704,did:plc:olol7qkyezzrxcynunk33d7g,did:plc:olol7qkyezzrxcynunk33d7g_3jwednr3bpd2m,2023-05-23 01:35:36.423,5658286,5764654,15888
7573709,did:plc:olol7qkyezzrxcynunk33d7g,did:plc:olol7qkyezzrxcynunk33d7g_3jxk7pyxndm23,2023-06-07 03:06:26.874,5341118,5447486,15888
7573717,did:plc:olol7qkyezzrxcynunk33d7g,did:plc:olol7qkyezzrxcynunk33d7g_3jyqzcgrdri2k,2023-06-22 13:25:18.236,2486089,2592457,15888
7573719,did:plc:olol7qkyezzrxcynunk33d7g,did:plc:olol7qkyezzrxcynunk33d7g_3jysq3lvbla23,2023-06-23 05:45:44.581,5341152,5447520,15888


In [9]:
creator_info = post_creators.get(1489942)
creator_id, created_at = creator_info
created_at.date()

datetime.date(2023, 4, 7)

In [23]:
producer_data = [
    {'timestamp': datetime.fromtimestamp(ts), 'producer_id': pid, 'embedding': emb}
    for ts, producers in producer_dynamic_features.items()
    for pid, emb in producers.items()
]
producer_dynamic_features_df = pd.DataFrame(producer_data)

In [28]:
producer_dynamic_features_df['producer_id'].value_counts()

producer_id
52475    108
20903    108
27092    108
8775     108
88511    108
        ... 
22862    108
16733    108
91587    108
7551     108
87116    108
Name: count, Length: 39729, dtype: int64

In [ ]:
{1678863600: {52475: array([-3.29911649e-01,  4.49524939e-01, -1.89922124e-01, -5.47734618e-01,
         -8.67967159e-02,  3.69976163e-01, -2.18309718e-03,  1.48269180e-02,
          6.13710191e-03, -7.99730420e-03,  2.42000818e-02, -6.45511672e-02,
          1.91325378e-02,  5.94373085e-02,  2.97789404e-04, -1.24324607e-02,
          5.71960397e-02, -2.21542269e-02,  1.19678430e-01,  2.57431287e-02,
         -1.06526017e-02,  5.21711633e-02,  4.20802869e-02,  3.04213222e-02,
          5.28707393e-02,  3.18310373e-02, -9.91340354e-02, -9.14021209e-02,
          1.41078383e-01,  7.27327242e-02, -1.27840504e-01, -2.30563041e-02,
          7.67583400e-02, -3.57548259e-02,  3.20819393e-02,  1.03408165e-01,
         -2.34360341e-02,  6.47874102e-02,  1.46832503e-02, -1.51609769e-02,
          8.57625306e-02, -3.00564840e-02,  1.49452053e-02, -3.58771011e-02,
          4.14313711e-02,  1.80791095e-02,  5.41538857e-02,  8.67804289e-02,
          5.28710969e-02,  5.88307603e-06,  7.94109255e-02,  1.05249360e-01,
         -5.27148917e-02, -2.12663393e-02, -3.27047892e-02,  4.68443632e-02,
         -5.49336560e-02, -8.01606625e-02, -8.44090581e-02,  3.33718769e-02,
         -1.02221899e-01, -6.83138743e-02,  6.44366294e-02,  3.36299203e-02],
        dtype=float32),
  16677: array([-0.28579423,  0.33537814, -0.10075051, -0.56730765, -0.07824453,
          0.44683632,  0.04620489, -0.05186317, -0.03248075, -0.03880691,